# LAB 04 — Text Classification

**Task:** binary sentiment classification trên IMDb Movie Reviews.

Notebook thực hiện dataset inspection, data leakage prevention, baseline TF-IDF + Logistic Regression, so sánh representation, n-gram, confusion matrix, error analysis và model interpretability.

Các mô hình dùng cùng một training split và test split. Test data không được dùng để fit vectorizer, Word2Vec hoặc classifier.

In [ ]:
from pathlib import Path
import os
import zipfile

archive_path = Path("/content/lab04.zip")
if not archive_path.exists():
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        pass
if archive_path.exists():
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall("/content")
project_path = Path("/content/lab04")
if project_path.exists() and (project_path / "preprocessing.py").exists():
    os.chdir(project_path)
%pip install -q -r requirements.txt

In [ ]:
import json
import time
import html
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from preprocessing import normalize_text, inspect_split, stratified_sample, remove_test_overlap, average_word2vec, binary_metrics

SEED = 42
TRAIN_SAMPLE_SIZE = 10000
TEST_SAMPLE_SIZE = 2500
WORD2VEC_SIZE = 100
OUTPUT_DIR = Path('.')
np.random.seed(SEED)

## 1. Load dataset

IMDb contém review text và nhãn sentiment: `0` là negative, `1` là positive. Notebook sử dụng train và test split do dataset cung cấp.

In [ ]:
dataset = load_dataset("stanfordnlp/imdb", "plain_text")
train_full = dataset["train"].to_pandas()[["text", "label"]]
test_full = dataset["test"].to_pandas()[["text", "label"]]

label_names = {0: "negative", 1: "positive"}
display(pd.DataFrame([
    {"Split": "Train", "Samples": len(train_full), "Classes": train_full["label"].nunique()},
    {"Split": "Test", "Samples": len(test_full), "Classes": test_full["label"].nunique()},
]))

## 2. Dataset inspection

Kiểm tra số samples, số classes, phân phối nhãn, document length, duplicate documents, missing text và missing labels trước khi train.

In [ ]:
inspection = pd.DataFrame([
    inspect_split(train_full, "train"),
    inspect_split(test_full, "test"),
])
display(inspection)

class_rows = []
for split_name, frame in [("train", train_full), ("test", test_full)]:
    counts = frame["label"].value_counts().sort_index()
    for label, count in counts.items():
        class_rows.append({
            "Split": split_name,
            "Class": label_names[int(label)],
            "Count": int(count),
            "Percentage": round(100 * count / len(frame), 2),
        })
class_distribution = pd.DataFrame(class_rows)
display(class_distribution)

fig, ax = plt.subplots(figsize=(7, 4))
plot_data = class_distribution.pivot(index="Class", columns="Split", values="Count")
plot_data.plot(kind="bar", ax=ax)
ax.set_title("IMDb class distribution")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Samples")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig("dataset_class_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

length_rows = []
for split_name, frame in [("train", train_full), ("test", test_full)]:
    word_lengths = frame["text"].fillna("").map(lambda value: len(simple_preprocess(str(value), deacc=True)))
    length_rows.append({
        "Split": split_name,
        "Mean words": round(word_lengths.mean(), 2),
        "Median words": round(word_lengths.median(), 2),
        "95th percentile": round(word_lengths.quantile(0.95), 2),
        "Maximum words": int(word_lengths.max()),
    })
display(pd.DataFrame(length_rows))

train_normalized = set(train_full["text"].map(normalize_text))
test_normalized = set(test_full["text"].map(normalize_text))
cross_split_duplicates = len(train_normalized.intersection(test_normalized))
display(pd.DataFrame([{"Exact normalized documents shared by official train and test": cross_split_duplicates}]))

### Nhận xét dataset

IMDb là bài toán binary sentiment classification. Cần kiểm tra tỷ lệ hai class thay vì mặc định dataset cân bằng. Độ dài review khác nhau đáng kể nên bước vectorization phải xử lý văn bản có số lượng từ khác nhau. Duplicate giữa train và test được kiểm tra riêng để tránh đánh giá lạc quan do mẫu trùng.

## 3. Build train, validation and test sets

Lấy sample có stratification để thực nghiệm chạy được trong môi trường lab. Test được lấy từ official test split. Những document trùng với official training data bị loại khỏi test sample. Validation được tách riêng từ training sample.

In [ ]:
train_sample = stratified_sample(train_full, TRAIN_SAMPLE_SIZE, random_state=SEED)
test_sample = stratified_sample(test_full, TEST_SAMPLE_SIZE, random_state=SEED + 1)
test_sample, removed_test_duplicates = remove_test_overlap(train_full, test_sample)

X_train, X_val, y_train, y_val = train_test_split(
    train_sample["text"].astype(str),
    train_sample["label"].astype(int),
    test_size=0.2,
    random_state=SEED,
    stratify=train_sample["label"],
)
X_test = test_sample["text"].astype(str)
y_test = test_sample["label"].astype(int)

split_summary = pd.DataFrame([
    {"Split": "train", "Samples": len(X_train), "Positive percentage": round(100 * y_train.mean(), 2)},
    {"Split": "validation", "Samples": len(X_val), "Positive percentage": round(100 * y_val.mean(), 2)},
    {"Split": "test", "Samples": len(X_test), "Positive percentage": round(100 * y_test.mean(), 2)},
])
display(split_summary)
display(pd.DataFrame([{"Duplicate test samples removed": removed_test_duplicates}]))

## 4. Prediction before experiment

Các giả thuyết ban đầu nằm trong `prediction.md`. Không dùng kết quả test để thay đổi giả thuyết ban đầu sau khi đã quan sát kết quả.

## 5. Experiment 1 — TF-IDF + Logistic Regression

Baseline dùng unigram TF-IDF. Vectorizer và classifier được fit trong pipeline trên training data. Validation được giữ riêng để quan sát threshold; test dùng để đánh giá cuối cùng.

In [ ]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 1),
        min_df=2,
        max_features=50000,
        sublinear_tf=True,
    )),
    ("classifier", LogisticRegression(max_iter=1500, random_state=SEED)),
])

start_time = time.perf_counter()
baseline.fit(X_train, y_train)
baseline_training_time = time.perf_counter() - start_time
baseline_test_predictions = baseline.predict(X_test)
baseline_validation_probabilities = baseline.predict_proba(X_val)[:, 1]
baseline_vocabulary_size = len(baseline.named_steps["tfidf"].vocabulary_)
baseline_metrics = binary_metrics(y_test, baseline_test_predictions)

display(pd.DataFrame([{
    "Model": "TF-IDF + Logistic Regression",
    "Training time seconds": round(baseline_training_time, 3),
    "Vocabulary size": baseline_vocabulary_size,
    **{key: round(value, 4) if isinstance(value, float) else value for key, value in baseline_metrics.items()},
}]))

## 6. Prediction threshold

Đánh giá trên validation probabilities tại threshold 0.5 và 0.8. Khi tăng threshold, mô hình cần bằng chứng mạnh hơn trước khi dự đoán positive.

In [ ]:
threshold_rows = []
for threshold in [0.5, 0.8]:
    validation_predictions = (baseline_validation_probabilities >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val, validation_predictions),
        "Precision": precision_score(y_val, validation_predictions, zero_division=0),
        "Recall": recall_score(y_val, validation_predictions, zero_division=0),
        "F1": f1_score(y_val, validation_predictions, zero_division=0),
        "Predicted positive": int(validation_predictions.sum()),
    })
threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results.round(4))

## 7. Experiment 2 — Average Word2Vec + Logistic Regression

Word2Vec chỉ được học trên review trong training split. Document representation là trung bình các word vectors xuất hiện trong review. Cách biểu diễn này tạo vector dense có chiều cố định nhưng không giữ thứ tự từ.

In [ ]:
training_tokens = [simple_preprocess(normalize_text(text), deacc=True) for text in X_train]

start_time = time.perf_counter()
word2vec_model = Word2Vec(
    sentences=training_tokens,
    vector_size=WORD2VEC_SIZE,
    window=5,
    min_count=2,
    workers=1,
    sg=1,
    epochs=10,
    seed=SEED,
)
X_train_w2v = average_word2vec(X_train.tolist(), word2vec_model, WORD2VEC_SIZE)
X_val_w2v = average_word2vec(X_val.tolist(), word2vec_model, WORD2VEC_SIZE)
X_test_w2v = average_word2vec(X_test.tolist(), word2vec_model, WORD2VEC_SIZE)
w2v_classifier = LogisticRegression(max_iter=1500, random_state=SEED)
w2v_classifier.fit(X_train_w2v, y_train)
w2v_training_time = time.perf_counter() - start_time
w2v_test_predictions = w2v_classifier.predict(X_test_w2v)
w2v_metrics = binary_metrics(y_test, w2v_test_predictions)
w2v_vocabulary_size = len(word2vec_model.wv)

display(pd.DataFrame([{
    "Model": "Average Word2Vec + Logistic Regression",
    "Training time seconds": round(w2v_training_time, 3),
    "Word2Vec vocabulary size": w2v_vocabulary_size,
    "Feature dimension": WORD2VEC_SIZE,
    **{key: round(value, 4) if isinstance(value, float) else value for key, value in w2v_metrics.items()},
}]))

## 8. Experiment 3 — TF-IDF + Linear SVM

Giữ unigram TF-IDF và thay Logistic Regression bằng Linear SVM để so sánh classifier trong cùng kiểu representation.

In [ ]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 1),
        min_df=2,
        max_features=50000,
        sublinear_tf=True,
    )),
    ("classifier", LinearSVC(C=1.0, random_state=SEED)),
])

start_time = time.perf_counter()
svm_pipeline.fit(X_train, y_train)
svm_training_time = time.perf_counter() - start_time
svm_test_predictions = svm_pipeline.predict(X_test)
svm_vocabulary_size = len(svm_pipeline.named_steps["tfidf"].vocabulary_)
svm_metrics = binary_metrics(y_test, svm_test_predictions)

display(pd.DataFrame([{
    "Model": "TF-IDF + Linear SVM",
    "Training time seconds": round(svm_training_time, 3),
    "Vocabulary size": svm_vocabulary_size,
    **{key: round(value, 4) if isinstance(value, float) else value for key, value in svm_metrics.items()},
}]))

## 9. Experiment 4 — Unigram vs unigram + bigram

Bigram cho phép biểu diễn cụm hai từ như `not good` thành feature riêng. Đổi lại, feature space lớn hơn và có thể chứa nhiều n-gram hiếm.

In [ ]:
bigram_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_features=80000,
        sublinear_tf=True,
    )),
    ("classifier", LogisticRegression(max_iter=1500, random_state=SEED)),
])

start_time = time.perf_counter()
bigram_pipeline.fit(X_train, y_train)
bigram_training_time = time.perf_counter() - start_time
bigram_test_predictions = bigram_pipeline.predict(X_test)
bigram_vocabulary_size = len(bigram_pipeline.named_steps["tfidf"].vocabulary_)
bigram_metrics = binary_metrics(y_test, bigram_test_predictions)

display(pd.DataFrame([
    {
        "Features": "Unigram",
        "Vocabulary size": baseline_vocabulary_size,
        "F1": baseline_metrics["F1"],
        "Training time seconds": baseline_training_time,
    },
    {
        "Features": "Unigram + bigram",
        "Vocabulary size": bigram_vocabulary_size,
        "F1": bigram_metrics["F1"],
        "Training time seconds": bigram_training_time,
    },
]).round(4))

## 10. Representation comparison and results.csv

Precision, recall và F1 được tính cho class positive. Accuracy được giữ lại nhưng không dùng như metric duy nhất. Training time gồm fit representation và classifier.

In [ ]:
model_records = [
    {
        "Model": "TF-IDF + Logistic Regression",
        "Representation": "TF-IDF unigram",
        "Classifier": "Logistic Regression",
        "Training_time_seconds": baseline_training_time,
        "Vocabulary_size": baseline_vocabulary_size,
        "Feature_Dimension": baseline_vocabulary_size,
        **baseline_metrics,
    },
    {
        "Model": "Average Word2Vec + Logistic Regression",
        "Representation": "Average Word2Vec",
        "Classifier": "Logistic Regression",
        "Training_time_seconds": w2v_training_time,
        "Vocabulary_size": w2v_vocabulary_size,
        "Feature_Dimension": WORD2VEC_SIZE,
        **w2v_metrics,
    },
    {
        "Model": "TF-IDF + Linear SVM",
        "Representation": "TF-IDF unigram",
        "Classifier": "Linear SVM",
        "Training_time_seconds": svm_training_time,
        "Vocabulary_size": svm_vocabulary_size,
        "Feature_Dimension": svm_vocabulary_size,
        **svm_metrics,
    },
    {
        "Model": "TF-IDF unigram + bigram + Logistic Regression",
        "Representation": "TF-IDF unigram + bigram",
        "Classifier": "Logistic Regression",
        "Training_time_seconds": bigram_training_time,
        "Vocabulary_size": bigram_vocabulary_size,
        "Feature_Dimension": bigram_vocabulary_size,
        **bigram_metrics,
    },
]
results = pd.DataFrame(model_records)
results.to_csv(OUTPUT_DIR / "results.csv", index=False, float_format="%.6f")
display(results.sort_values("F1", ascending=False).round(4))

## 11. Confusion matrix

Confusion matrix được vẽ cho baseline TF-IDF + Logistic Regression. Trục dọc là nhãn thật, trục ngang là nhãn dự đoán.

In [ ]:
cm = confusion_matrix(y_test, baseline_test_predictions, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
image = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.figure.colorbar(image, ax=ax)
ax.set(
    xticks=np.arange(2),
    yticks=np.arange(2),
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"],
    ylabel="Actual label",
    xlabel="Predicted label",
    title="Confusion Matrix — TF-IDF + Logistic Regression",
)
threshold_value = cm.max() / 2 if cm.size else 0
for row in range(cm.shape[0]):
    for column in range(cm.shape[1]):
        ax.text(column, row, format(cm[row, column], "d"), ha="center", va="center", color="white" if cm[row, column] > threshold_value else "black")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

display(pd.DataFrame([{
    "True positives": int(cm[1, 1]),
    "False positives": int(cm[0, 1]),
    "False negatives": int(cm[1, 0]),
    "True negatives": int(cm[0, 0]),
}]))

## 12. Word order and negation

Average embeddings không mã hóa thứ tự token. Hai câu có cùng tập token sẽ có cùng vector trung bình. Với `not good`, từ `not` vẫn có vector riêng nhưng phép averaging không biểu diễn trực tiếp phạm vi tác động của phủ định lên `good`.

In [ ]:
examples = [
    "this movie is good",
    "this movie is not good",
    "this movie is good not",
    "this movie is very good",
]
example_vectors = average_word2vec(examples, word2vec_model, WORD2VEC_SIZE)

def cosine_similarity(first, second):
    first_norm = np.linalg.norm(first)
    second_norm = np.linalg.norm(second)
    if first_norm == 0 or second_norm == 0:
        return 0.0
    return float(np.dot(first, second) / (first_norm * second_norm))

word_order_rows = [
    {
        "Sentence A": examples[0],
        "Sentence B": examples[1],
        "Cosine similarity": cosine_similarity(example_vectors[0], example_vectors[1]),
        "Interpretation": "Difference is the added negation token",
    },
    {
        "Sentence A": examples[1],
        "Sentence B": examples[2],
        "Cosine similarity": cosine_similarity(example_vectors[1], example_vectors[2]),
        "Interpretation": "Same tokens in a different order",
    },
    {
        "Sentence A": examples[0],
        "Sentence B": examples[3],
        "Cosine similarity": cosine_similarity(example_vectors[0], example_vectors[3]),
        "Interpretation": "Difference is the intensifier token",
    },
]
display(pd.DataFrame(word_order_rows).round(4))

## 13. Inspect Logistic Regression weights

Với nhãn `1` là positive, hệ số dương đóng góp theo hướng positive và hệ số âm đóng góp theo hướng negative trong linear score.

In [ ]:
feature_names = baseline.named_steps["tfidf"].get_feature_names_out()
weights = baseline.named_steps["classifier"].coef_[0]
weight_order = np.argsort(weights)
negative_features = pd.DataFrame({
    "Feature": feature_names[weight_order[:15]],
    "Weight": weights[weight_order[:15]],
}).sort_values("Weight")
positive_features = pd.DataFrame({
    "Feature": feature_names[weight_order[-15:][::-1]],
    "Weight": weights[weight_order[-15:][::-1]],
}).sort_values("Weight", ascending=False)

display(positive_features.round(4).rename(columns={"Feature": "Top positive feature"}))
display(negative_features.round(4).rename(columns={"Feature": "Top negative feature"}))

## 14. Error analysis

Lấy tối thiểu ba mẫu dự đoán đúng và ba mẫu dự đoán sai từ test set. Nhóm lỗi dựa trên heuristic lexical và cần được kiểm tra thủ công trước khi xem là nguyên nhân cuối cùng.

In [ ]:
error_frame = pd.DataFrame({
    "Text": X_test.reset_index(drop=True),
    "True label": y_test.reset_index(drop=True),
    "Predicted label": pd.Series(baseline_test_predictions),
})
error_frame["Correct"] = error_frame["True label"] == error_frame["Predicted label"]

vectorizer = baseline.named_steps["tfidf"]
classifier = baseline.named_steps["classifier"]
feature_names = vectorizer.get_feature_names_out()
feature_weights = classifier.coef_[0]
training_vocabulary = set(vectorizer.vocabulary_)


def candidate_category(text):
    lowered = normalize_text(text).lower()
    tokens = simple_preprocess(lowered, deacc=True)
    if any(phrase in lowered for phrase in ["yeah right", "as if", "what a joke", "laughable", "so-called"]):
        return "sarcasm"
    if any(term in lowered for term in ["not ", "no ", "never", "neither", "nor ", "without", "isn't", "wasn't", "weren't", "didn't", "doesn't", "can't", "couldn't", "won't"]):
        return "negation"
    if len(tokens) <= 30:
        return "short text"
    content_tokens = [token for token in tokens if len(token) > 2]
    if content_tokens:
        out_of_vocabulary_ratio = sum(token not in training_vocabulary for token in content_tokens) / len(content_tokens)
        if out_of_vocabulary_ratio >= 0.25:
            return "rare word"
    if any(character.isdigit() for character in text):
        return "domain-specific term"
    return "lexical ambiguity"


def get_feature_evidence(text, direction, limit=3):
    row = vectorizer.transform([text])
    contributions = [
        (feature_names[index], float(value * feature_weights[index]))
        for index, value in zip(row.indices, row.data)
        if (value * feature_weights[index] > 0 if direction == "positive" else value * feature_weights[index] < 0)
    ]
    contributions.sort(key=lambda item: item[1], reverse=(direction == "positive"))
    return contributions[:limit]


def format_evidence(items):
    if not items:
        return "no strong matched feature"
    return ", ".join(f"{term} ({contribution:+.3f})" for term, contribution in items)


def explain_sample(row):
    true_label = label_names[int(row["True label"])]
    predicted_label = label_names[int(row["Predicted label"])]
    predicted_direction = "positive" if predicted_label == "positive" else "negative"
    true_direction = "positive" if true_label == "positive" else "negative"
    predicted_features = get_feature_evidence(row["Text"], predicted_direction)

    if row["Correct"]:
        return (
            f"Correct prediction. Features contributing toward {predicted_label}: "
            f"{format_evidence(predicted_features)}. Check the full review to confirm these terms "
            "express sentiment rather than incidental topic or plot information. Contributions "
            "describe the linear model score and do not prove causation."
        )

    true_features = get_feature_evidence(row["Text"], true_direction)
    return (
        f"Incorrect prediction: true label is {true_label}, but the model predicts {predicted_label}. "
        f"Features pushing toward the prediction: {format_evidence(predicted_features)}. "
        f"Features pushing toward the true label: {format_evidence(true_features)}. "
        "Read the full review to check whether negation, contrast, sarcasm, or wider context changes "
        "the sentiment. These feature contributions are evidence for inspection, not a confirmed cause."
    )


correct_count = int(error_frame["Correct"].sum())
incorrect_count = int((~error_frame["Correct"]).sum())
correct_examples = error_frame[error_frame["Correct"]].sample(n=min(3, correct_count), random_state=SEED)
incorrect_examples = error_frame[~error_frame["Correct"]].sample(n=min(3, incorrect_count), random_state=SEED)
selected_errors = pd.concat([correct_examples, incorrect_examples], axis=0).copy()
selected_errors["Analysis category"] = selected_errors["Text"].map(candidate_category)
selected_errors["Analysis"] = selected_errors.apply(explain_sample, axis=1)
selected_errors["True label"] = selected_errors["True label"].map(label_names)
selected_errors["Predicted label"] = selected_errors["Predicted label"].map(label_names)
selected_errors["Text excerpt"] = selected_errors["Text"].map(lambda text: text[:320] + ("…" if len(text) > 320 else ""))

output_columns = ["Text excerpt", "True label", "Predicted label", "Analysis category", "Analysis"]
with pd.option_context("display.max_colwidth", 240, "display.width", 220):
    display(selected_errors[output_columns])

markdown_lines = [
    "# Error Analysis",
    "",
    "Baseline: TF-IDF + Logistic Regression. Examples are drawn from the held-out test split.",
    "",
    "Three correct predictions and three incorrect predictions are sampled where enough examples are available.",
    "The category is an automatic candidate, not a confirmed diagnosis. Feature contributions show which TF-IDF terms affect the linear score; manually inspect the complete review before assigning a final explanation.",
    "",
    "| Text excerpt | True label | Predicted label | Candidate category | Analysis |",
    "|---|---|---|---|---|",
]
for _, row in selected_errors.iterrows():
    safe_values = [
        str(row["Text excerpt"]).replace("|", "\\|").replace("\n", " "),
        str(row["True label"]),
        str(row["Predicted label"]),
        str(row["Analysis category"]),
        str(row["Analysis"]).replace("|", "\\|").replace("\n", " "),
    ]
    markdown_lines.append("| " + " | ".join(safe_values) + " |")
Path("error_analysis.md").write_text("\n".join(markdown_lines) + "\n", encoding="utf-8")


## 15. Reflection

Tổng kết được tạo từ kết quả của lần chạy hiện tại. Đánh giá dựa trên nhiều metric, không mặc định mô hình có F1 cao nhất luôn phù hợp cho mọi ứng dụng.

In [ ]:
results_indexed = results.set_index("Model")
model_f1 = results_indexed["F1"]
best_model_name = model_f1.idxmax()
best_f1 = float(model_f1.max())

w2v_f1 = float(results_indexed.loc["Average Word2Vec + Logistic Regression", "F1"])
tfidf_f1 = float(results_indexed.loc["TF-IDF + Logistic Regression", "F1"])
bigram_f1 = float(results_indexed.loc["TF-IDF unigram + bigram + Logistic Regression", "F1"])
svm_f1 = float(results_indexed.loc["TF-IDF + Linear SVM", "F1"])

if tfidf_f1 >= w2v_f1:
    representation_conclusion = "Trong lần chạy này, TF-IDF + Logistic Regression có F1 không thấp hơn Average Word2Vec + Logistic Regression. Điều này phù hợp với giả thuyết rằng lexical feature có tín hiệu hữu ích cho sentiment classification."
else:
    representation_conclusion = "Trong lần chạy này, Average Word2Vec + Logistic Regression có F1 cao hơn TF-IDF + Logistic Regression. Kết quả cần được xem cùng training time, sampling và các ví dụ lỗi; không thể suy ra Word2Vec luôn tốt hơn."

if bigram_f1 >= tfidf_f1:
    bigram_conclusion = "Unigram + bigram đạt F1 không thấp hơn unigram trong lần chạy này. Cụm từ có thể bổ sung tín hiệu, nhưng cần cân nhắc vocabulary size và training time."
else:
    bigram_conclusion = "Unigram đạt F1 cao hơn unigram + bigram trong lần chạy này. Feature space lớn hơn không bảo đảm metric tốt hơn."

reflection_text = f"""# Reflection

## Experimental setup

- Dataset: IMDb Movie Reviews.
- Training samples: {len(X_train)}.
- Validation samples: {len(X_val)}.
- Test samples: {len(X_test)}.
- Random seed: {SEED}.
- Baseline test accuracy: {baseline_metrics['Accuracy']:.4f}.
- Baseline test precision: {baseline_metrics['Precision']:.4f}.
- Baseline test recall: {baseline_metrics['Recall']:.4f}.
- Baseline test F1: {baseline_metrics['F1']:.4f}.

## Representation comparison

{representation_conclusion}

Average Word2Vec creates a dense fixed-size vector but averaging removes word order and can dilute the effect of negation. TF-IDF preserves separate lexical features and can represent selected phrases when n-grams are enabled.

## N-gram comparison

{bigram_conclusion}

Unigram vocabulary size: {baseline_vocabulary_size}.

Unigram plus bigram vocabulary size: {bigram_vocabulary_size}.

## Classifier comparison

F1 for TF-IDF + Logistic Regression: {tfidf_f1:.4f}.

F1 for TF-IDF + Linear SVM: {svm_f1:.4f}.

Best F1 in this experiment: {best_model_name} at {best_f1:.4f}.

## Data leakage

TF-IDF was fit inside a training-only pipeline. Validation and test documents were transformed using the vocabulary and inverse document frequencies learned from training data. Word2Vec was also trained only from training text. The held-out test split was not used for fitting.

## Evaluation beyond accuracy

Precision and recall describe different errors. False positives are negative reviews classified as positive; false negatives are positive reviews classified as negative. For the baseline, FP = {baseline_metrics['FP']} and FN = {baseline_metrics['FN']}. The larger of these two counts identifies the more frequent error type in this test sample. Precision is especially sensitive to false positives, while recall is especially sensitive to false negatives. Confusion matrix and selected examples are required to understand the error pattern. A higher accuracy alone is not sufficient evidence of a better classifier.

## Limitations

The experiment uses a stratified sample for practical runtime. Results can change with random seed, sample size, preprocessing, hyperparameters and the dataset version. Error categories are heuristic labels that require manual confirmation. The scores are not claims about every sentiment dataset.

## Prediction review

- P1: check the baseline metrics above and compare against the other representations.
- P2: compare the F1 values of TF-IDF and Average Word2Vec.
- P3: compare unigram and unigram plus bigram vocabulary sizes and F1.
- P4: stopwords were preserved so that terms such as `not` remain available to the model.
- P5: compare accuracy, precision, recall and F1 in `results.csv` to determine whether the metrics rank models differently.
"""
Path("reflection.md").write_text(reflection_text, encoding="utf-8")

## 16. Final checklist

Notebook outputs gồm `results.csv`, `confusion_matrix.png`, `dataset_class_distribution.png`, `error_analysis.md` và `reflection.md`. Kiểm tra tất cả file sau khi chạy hoàn tất trước khi commit lên GitHub.

In [ ]:
import zipfile
from pathlib import Path

project_root = OUTPUT_DIR.resolve()
completed_archive = project_root.parent / "lab04_completed.zip"
with zipfile.ZipFile(completed_archive, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in project_root.rglob("*"):
        if file_path.is_file() and "__pycache__" not in file_path.parts and ".ipynb_checkpoints" not in file_path.parts and file_path.suffix != ".pyc":
            archive.write(file_path, Path("lab04") / file_path.relative_to(project_root))

try:
    from google.colab import files
    files.download(str(completed_archive))
except ImportError:
    pass